# Практика: воспроизводимый preprocessing pipeline


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


**Центральная идея:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## 1. Контракт функции

Запишите сигнатуру и docstring, пока тело может быть заглушкой.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def preprocess_customers(orders_df,customers_df,payments_df):
    '''Return customer features and ordered audit log.'''
    # TODO
    ...
assert preprocess_customers.__doc__


## 2. Копии и валидация

Реализуйте начало функции: копии, schema/key/value checks, первый лог.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def preprocess_customers(orders_df, customers_df, payments_df):
    log=[]
    o,c,p=orders_df.copy(),customers_df.copy(),payments_df.copy()
    required_o={"order_id","customer_id","order_purchase_timestamp","order_delivered_customer_date"}
    required_c={"customer_id","customer_state"}
    required_p={"order_id","payment_type","payment_value"}
    for frame,required,name in ((o,required_o,"orders"),(c,required_c,"customers"),(p,required_p,"payments")):
        missing=required-set(frame.columns)
        if missing: raise KeyError(f"{name} missing columns: {sorted(missing)}")
    if o["order_id"].duplicated().any() or p["order_id"].duplicated().any(): raise ValueError("duplicate order_id")
    if o["order_purchase_timestamp"].isna().any(): raise ValueError("missing order_purchase_timestamp")
    if p["payment_value"].isna().any() or p["payment_value"].lt(0).any(): raise ValueError("invalid payment_value")
    log.append("validated input contracts")
    x=o.merge(p,on="order_id",validate="one_to_one").merge(c[["customer_id","customer_state"]],on="customer_id",validate="many_to_one")
    log.append("merged orders, payments, customers")
    x["days_to_deliver"]=(x["order_delivered_customer_date"]-x["order_purchase_timestamp"]).dt.days
    x["is_card"]=(x["payment_type"]=="credit_card").astype(int)
    ref=x["order_purchase_timestamp"].max()
    features=x.groupby("customer_id").agg(last_purchase=("order_purchase_timestamp","max"),Frequency=("order_id","nunique"),Monetary=("payment_value","sum"),share_card=("is_card","mean"),avg_days_to_deliver=("days_to_deliver","mean"),customer_state=("customer_state","first")).reset_index()
    features["Recency"]=(ref-features["last_purchase"]).dt.days
    features=features[["customer_id","Recency","Frequency","Monetary","share_card","avg_days_to_deliver","customer_state"]]
    log.append("built customer RFM plus features")
    return features,log

features,log=preprocess_customers(orders,customers,payments)
assert log[0]=="validated input contracts"
assert "Recency" not in orders


## 3. Форма результата

Проверьте одну строку клиента и семь ожидаемых полей.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
required={"customer_id","Recency","Frequency","Monetary","share_card","avg_days_to_deliver","customer_state"}
assert required==set(features.columns)
assert len(features)==778 and features["customer_id"].is_unique


## 4. Инварианты RFM

Сверьте сумму заказов, денег и границы.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
checks={"orders":None,"money":None,"recency":None,"share_card":None}  # TODO
assert set(checks.values())=={True}


## 5. Негативный тест

Подайте отрицательную оплату и проверьте raise.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
bad=payments.copy(); bad.loc[bad.index[0],"payment_value"]=-10
caught=""
try: preprocess_customers(orders,customers,bad)
except ValueError as e: caught=str(e)
assert "payment_value" in caught


## 6. Повторяемость

Запустите дважды и сравните DataFrame и лог.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
features2,log2=preprocess_customers(orders,customers,payments)
same=None  # TODO
assert same is True and log2==log


## 7. Preview артефакта

Сохраните первые 25 строк и прочитайте обратно.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
preview_path=Path("features_preview.csv")
# TODO
loaded_preview=None  # TODO
assert preview_path.exists() and len(loaded_preview)==25
assert list(loaded_preview.columns)==list(features.columns)


## 8. Acceptance и отчёт

Закройте пять критериев и напишите итог без обещания модели.

**Зачем:** Итоговый pipeline объединяет копирование, контракт, join, признаки, агрегацию и аудит в одной повторяемой функции. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
acceptance={"contract":None,"rfm":None,"extras":None,"audit":None,"preview":None}  # TODO
REPORT=""  # TODO
assert set(acceptance.values())=={True}
assert len(REPORT)>=320 and "churn" not in REPORT.lower()
